In [1]:
import torch
from typing import Tuple, List, Dict



class NodeMRREvaluator:
    r"""
    Node-level MRR evaluator (ogbl-citation2 logic) — supports sparse test_neg
    -------------------------------------------------------------------------
    Parameters
    ----------
    deg        : LongTensor [N]   — out-degree of every node
    test_pos   : LongTensor [2,E] — (u,v) sorted by (u,v)
    test_neg   : LongTensor [2,A*K]
                 row0 = u repeated K times for each active node (ascending u)
                 row1 = negative target nodes, same order
    use_fp16   : bool             — compute in fp16 to save memory
    max_memory : float (GB)       — GPU memory cap per chunk
    """

    # --------------------------------------------------  constructor
    def __init__(
        self,
        deg:        torch.LongTensor,
        test_pos:   torch.LongTensor,
        test_neg:   torch.LongTensor,
        use_fp16:   bool  = False,
        max_memory: float = 4.0
    ):
        # ---------- basic ----------
        assert deg.dim() == 1, "`deg` must be 1-D"
        self.N = deg.numel()
        self.dtype  = torch.float16 if use_fp16 else torch.float32
        self.bytes  = 2 if use_fp16 else 4
        self.max_bytes = int(max_memory * (1024 ** 3))

        # ---------- active nodes & K ----------
        self.active_u = torch.nonzero(deg > 0, as_tuple=False).flatten()  # [A]
        self.A = self.active_u.numel()
        total_neg = test_neg.size(1)
        assert total_neg % self.A == 0, "test_neg length must be A*K"
        self.K = total_neg // self.A

        # ---------- shape & ordering checks ----------
        assert test_neg.shape == (2, self.A * self.K), "test_neg shape mismatch"
        assert test_pos.shape[0] == 2, "test_pos must be [2,E]"

        # (1) test_neg row-0 must be u repeated K times, ascending u
        u_in_neg = test_neg[0].view(self.A, self.K)[:, 0]
        assert torch.equal(u_in_neg, self.active_u), \
               "test_neg row-0 must list active nodes in ascending order, each K times"

        # (2) test_pos sorted by (u,v)
        u_pos, v_pos = test_pos
        ok_order = ((u_pos[1:] - u_pos[:-1]) > 0) | \
                   ((u_pos[1:] == u_pos[:-1]) & (v_pos[1:] >= v_pos[:-1]))
        assert bool(ok_order.all()), "test_pos is not sorted by (u,v)"

        # (3) deg consistency
        deg_from_pos = torch.bincount(u_pos, minlength=self.N)
        assert torch.equal(deg_from_pos, deg), "`deg` inconsistent with test_pos"

        # ---------- store CPU tensors ----------
        self.deg_all     = deg.cpu()                       # [N]
        self.deg_active  = self.deg_all[self.active_u]     # [A]
        self.test_pos    = test_pos.cpu()                  # [2,E]
        self.neg_dst     = test_neg[1].cpu().view(self.A, self.K)  # [A,K]

        # prefix sum over all N nodes
        self.pos_pref = torch.zeros(self.N + 1, dtype=torch.long)
        torch.cumsum(self.deg_all, dim=0, out=self.pos_pref[1:])    # [N+1]

        # ---------- plan chunks over active rows ----------
        self.chunk_info: List[Dict] = self._plan_chunks()

    # --------------------------------------------------  chunk planner
    def _plan_chunks(self) -> List[Dict]:
        chunks = []
        l = 0
        B = 0         # rows in current chunk
        d_max = 0
        cur_bytes = 0
        for i, d in enumerate(self.deg_active.tolist()):
            B_new = B + 1
            d_max_new = max(d_max, d)
            mem_new = (self.K + d_max_new) * B_new * self.bytes
            if mem_new > self.max_bytes and B > 0:
                chunks.append(self._make_chunk(l, i, B, d_max))
                l, B, d_max, cur_bytes = i, 0, 0, 0
                B_new, d_max_new = 1, d
            B, d_max = B_new, d_max_new
        chunks.append(self._make_chunk(l, self.A, B, d_max))
        return chunks

    @staticmethod
    def _make_chunk(l, r, B, d_max):
        mem_MB = (d_max + NodeMRREvaluator.K if hasattr(NodeMRREvaluator, 'K') else 0) \
                 * B * (NodeMRREvaluator.bytes if hasattr(NodeMRREvaluator, 'bytes') else 4) / (1024 ** 2)
        return {"slice": slice(l, r), "B": B, "max_deg": d_max, "mem_MB": mem_MB}

    # --------------------------------------------------  evaluator
    @torch.no_grad()
    def evaluate(
        self,
        prob_pos: torch.Tensor,      # [2,E]
        prob_neg: torch.Tensor       # [2,A*K]
    ) -> Tuple[float, float]:

        # dtype align & flatten
        pos_scores = prob_pos[1].to(self.dtype).cpu()                   # [E]
        neg_scores = prob_neg[1].to(self.dtype).cpu().view(self.A, self.K)

        rr_opt_sum = 0.0
        rr_pes_sum = 0.0
        device = "cuda" if torch.cuda.is_available() else "cpu"

        for ch in self.chunk_info:
            sl = ch["slice"]                     # rows in [l,r)
            B  = ch["B"]
            d_max = ch["max_deg"]

            u_chunk = self.active_u[sl]          # [B]

            # ---- negatives ------------------------------------------------
            neg_s = neg_scores[sl].to(device)            # [B,K]
            neg_sorted = torch.sort(neg_s, dim=1).values # [B,K]

            # ---- positives -------------------------------------------------
            pos_start = self.pos_pref[u_chunk[0]]
            pos_end   = self.pos_pref[u_chunk[-1] + 1]
            pos_flat  = pos_scores[pos_start:pos_end].to(device)       # [sum deg_chunk]

            deg_chunk = self.deg_active[sl].to(device)                 # [B]
            mask = torch.arange(d_max, device=device).unsqueeze(0) < deg_chunk.unsqueeze(1)  # [B,d_max]
            pos_padded = torch.full((B, d_max), -float("inf"),
                                    dtype=self.dtype, device=device)
            pos_padded[mask] = pos_flat

            # ---- batched searchsorted -------------------------------------
            inds_opt = torch.searchsorted(neg_sorted, pos_padded, right=True)
            inds_pes = torch.searchsorted(neg_sorted, pos_padded, right=False)

            rank_opt = (self.K - inds_opt + 1).to(torch.float32)
            rank_pes = (self.K - inds_pes + 1).to(torch.float32)

            rr_opt_sum += (mask * (1.0 / rank_opt)).sum().item()
            rr_pes_sum += (mask * (1.0 / rank_pes)).sum().item()

        E = prob_pos.size(1)
        return rr_opt_sum / E, rr_pes_sum / E



class NodeMRREvaluator_bf:
    r"""
    逐节点 MRR —— 纯 Python 循环版本（对拍用，适配 test_neg = [2, A*K]）
    --------------------------------------------------------------------
    参数
    ----
    deg        : [N]  每节点正边数（LongTensor, CPU）
    test_pos   : [2,E]  (u,v)  按 (u,v) 排序
    test_neg   : [2,A*K]  (u,w)  仅包含 deg(u)>0 的节点，u 为主排序键
    use_fp16   : bool  （对 BF 版几乎无影响）
    max_memory : 保留兼容接口，BF 版不使用
    """

    # -------------------------------------------------------- ctor
    def __init__(
        self,
        deg:        torch.LongTensor,
        test_pos:   torch.LongTensor,
        test_neg:   torch.LongTensor,
        use_fp16:   bool  = False,
        max_memory: float = 4.0
    ):
        # ---------- 基本信息 ----------
        self.N = deg.numel()
        self.E = test_pos.size(1)

        # ---------- 解析 test_neg ----------
        u_neg_all = test_neg[0].tolist()           # 按 (u重复K次) 排列
        first_u = u_neg_all[0]
        K = 0
        for val in u_neg_all:       # 统计 K
            if val != first_u:
                break
            K += 1
        self.K = K

        # 活跃节点列表（升序）
        self.active_u: List[int] = sorted(set(u_neg_all))
        self.A = len(self.active_u)

        # ---------- 构造 node → slice 映射 ----------
        # test_neg 结构:  u0(×K), u1(×K), ...
        self.neg_slice_by_node: Dict[int, slice] = {}
        for idx, u in enumerate(self.active_u):
            start = idx * self.K
            self.neg_slice_by_node[u] = slice(start, start + self.K)

        # ---------- 正边源节点列表 ----------
        self.pos_u = test_pos[0].tolist()          # [E]

    # -------------------------------------------------------- evaluate
    @torch.no_grad()
    def evaluate(
        self,
        prob_pos: torch.Tensor,     # [2,E]
        prob_neg: torch.Tensor      # [2,A*K]
    ) -> Tuple[float, float]:

        pos_scores = prob_pos[1].tolist()                 # [E]
        neg_scores_flat = prob_neg[1].tolist()            # [A*K]

        rr_opt_sum = 0.0
        rr_pes_sum = 0.0

        for u, pos_s in zip(self.pos_u, pos_scores):
            sl = self.neg_slice_by_node[u]                # 该节点负边区间
            neg_list = neg_scores_flat[sl]

            gt_cnt = 0    # >  pos_s
            ge_cnt = 0    # >= pos_s
            for neg_s in neg_list:
                if neg_s > pos_s:
                    gt_cnt += 1
                    ge_cnt += 1
                elif neg_s == pos_s:
                    ge_cnt += 1

            rank_opt = gt_cnt + 1       # optimistic
            rank_pes = ge_cnt + 1       # pessimistic

            rr_opt_sum += 1.0 / rank_opt
            rr_pes_sum += 1.0 / rank_pes

        mrr_opt = rr_opt_sum / self.E
        mrr_pes = rr_pes_sum / self.E
        return mrr_opt, mrr_pes




In [2]:


import torch, random
from typing import List

# ---------------------------------------------------------------
# 请确保已在同一脚本 / notebook 中定义：
#   - NodeMRREvaluator        (高效版本，适配 test_neg=[2,A*K])
#   - NodeMRREvaluator_bf     (笨办法版本，对拍用，亦适配新格式)
# ---------------------------------------------------------------

def test_mrr_evaluators(
    N: int = 500,              # 节点总数
    K: int = 1000,             # 每有出度节点负样本数
    repeats: int = 10,         # 随机测试轮数
    max_deg: int = 20,         # 节点最大正边数
    use_fp16: bool = True,
    max_memory: float = 2.0,   # GiB
    tol_fp32: float = 1e-8,
    tol_fp16: float = 1e-3
) -> bool:
    """
    随机生成图，对拍 NodeMRREvaluator 与 NodeMRREvaluator_bf。
    满足：只为有出度节点构造负样本，test_neg 排布 (u 重复 K 次, v)。
    """
    torch.manual_seed(0)
    random.seed(0)

    # ---------- 1) 随机 out-degree ----------------------------
    # 让至少一个节点度 = max_deg，部分节点度 = 0
    deg_list = torch.randint(0, max_deg, (N,), dtype=torch.long)
    random_idx = random.randrange(N)
    deg_list[random_idx] = max_deg
    E = int(deg_list.sum().item())
    active_u: List[int] = torch.nonzero(deg_list > 0, as_tuple=False).flatten().tolist()
    A = len(active_u)

    # ---------- 2) 构造正边 (u,v) -----------------------------
    pos_u = torch.empty(E, dtype=torch.long)
    pos_v = torch.empty(E, dtype=torch.long)
    ofs = 0
    for u in range(N):
        d = deg_list[u].item()
        if d == 0:
            continue
        # 在除 u 外的 N-1 节点里无放回抽 d 个
        vs = torch.randperm(N - 1)[:d]
        vs[vs >= u] += 1
        pos_u[ofs:ofs + d] = u
        pos_v[ofs:ofs + d] = vs
        ofs += d
    # 按 (u,v) 排序：键 = u*N + v
    sort_idx = torch.argsort(pos_u * N + pos_v)
    pos_u, pos_v = pos_u[sort_idx], pos_v[sort_idx]
    test_pos = torch.stack([pos_u, pos_v], dim=0)            # [2,E]

    # ---------- 3) 构造负样本 -------------------------------
    neg_u = torch.arange(A, dtype=torch.long).repeat_interleave(K)
    neg_u = torch.tensor(active_u, dtype=torch.long).repeat_interleave(K)  # [A*K]
    neg_v = torch.empty(A * K, dtype=torch.long)
    for i, u in enumerate(active_u):
        start = i * K
        # 随机采样目标，避免 v==u
        vs = torch.randint(0, N - 1, (K,), dtype=torch.long)
        vs[vs >= u] += 1
        neg_v[start:start + K] = vs
    test_neg = torch.stack([neg_u, neg_v], dim=0)            # [2,A*K]

    # ---------- 4) 初始化评估器 ------------------------------
    eval_fast = NodeMRREvaluator(
        deg        = deg_list,
        test_pos   = test_pos,
        test_neg   = test_neg,
        use_fp16   = use_fp16,
        max_memory = max_memory
    )
    eval_bf = NodeMRREvaluator_bf(
        deg        = deg_list,
        test_pos   = test_pos,
        test_neg   = test_neg,
        use_fp16   = use_fp16
    )

    # ---------- 5) 多轮对拍 --------------------------------
    tol = tol_fp16 if use_fp16 else tol_fp32
    for rnd in range(repeats):
        pos_scores = torch.rand(E, dtype=eval_fast.dtype)
        neg_scores = torch.rand(A * K, dtype=eval_fast.dtype)

        prob_pos = torch.stack([pos_u, pos_scores], dim=0)   # [2,E]
        prob_neg = torch.stack([neg_u, neg_scores], dim=0)   # [2,A*K]

        fast_opt, fast_pes = eval_fast.evaluate(prob_pos, prob_neg)
        bf_opt,   bf_pes   = eval_bf.evaluate(prob_pos, prob_neg)

        print(abs(fast_opt - bf_opt), f"round {rnd}: optimistic diff {fast_opt} and {bf_opt}")
        print(abs(fast_pes - bf_pes), f"round {rnd}: pessimistic diff {fast_pes} and {bf_pes}")

    print(f"All {repeats} rounds passed ✔ (N={N}, A={A}, E={E})")
    return True




In [4]:
random.seed(1)
torch.manual_seed(1)
test_mrr_evaluators(
    N = 100,              # 节点总数
    K = 50,             # 每有出度节点负样本数
    repeats = 20,         # 随机测试轮数
    max_deg = 30,         # 节点最大正边数
    use_fp16 = True,
    max_memory = 2.0,   # GiB
    tol_fp32 = 1e-8,
    tol_fp16 = 1e-3
)

2.7323307505433547e-09 round 0: optimistic diff 0.08744618437436792 and 0.08744618164203717
4.6327285063707535e-10 round 0: pessimistic diff 0.08663319214262237 and 0.08663319167934952
2.8237083504745186e-09 round 1: optimistic diff 0.09288102924630866 and 0.09288103207001701
1.8624710246273324e-09 round 1: pessimistic diff 0.09258640128817759 and 0.09258640315064862
6.334576183042451e-09 round 2: optimistic diff 0.08848328821867416 and 0.08848328188409797
4.471723408228101e-10 round 2: pessimistic diff 0.08815005404277913 and 0.08815005359560679
9.02480423725649e-10 round 3: optimistic diff 0.09110063966423948 and 0.0911006405667199
9.119002747359772e-09 round 3: pessimistic diff 0.09085780270277104 and 0.09085779358376829
1.1232220431534046e-09 round 4: optimistic diff 0.08625866541198927 and 0.08625866653521132
5.393829966005015e-10 round 4: pessimistic diff 0.08618832683871865 and 0.08618832737810164
4.563081579256334e-10 round 5: optimistic diff 0.08386521817796824 and 0.083865217

True